# 03: Regression Modelling
***
**Author:** Zain Zabidi
**Date:** February 2026

**Objective:** To build and evaluate machine learning models that can predict a suburb's `housing_stress_index`. We will transition from simply describing the data to creating a predictive tool that can be used by the final Flask API.

## 1. Data Preparation and Feature Selection
In this section, we pick the specific variables (features) that will help our model predict housing stress across different suburbs.

### 1.1 Loading the Spatial Dataset
We use `geopandas` to load our processed data. We are using absolute path logic to ensure the notebook can find the file regardless of the local environment setup.

In [11]:
import geopandas as gpd
import os

# Set path to the processed geopackage
base_path = os.getcwd()
geo_path = os.path.join(base_path, '..', 'data', 'processed', 'wa_census_spatial_2021.gpkg')

# Load the dataset
df = gpd.read_file(geo_path)

# Display success message and suburb count
print(f"Data loaded successfully. Total suburbs: {len(df)}")

Data loaded successfully. Total suburbs: 1699


### 1.2 Defining the Target and Predictors
We now define our **Target** (the value we want to predict) and our **Features** (the data points used to make that prediction). We use the `pandas` library to filter our dataset and remove any rows with missing values. This ensures our machine learning model learns from a clean and complete dataset.

In [12]:
import pandas as pd

# Define the target variable
target = 'housing_stress_index'

# Define the predictive features
features = [
    'avg_weekly_income', 
    'avg_weekly_mortgage', 
    'avg_weekly_rent',
    'mining_concentration_ratio',
    'unemployment_rate',
    'avg_household_size',
    'income_rent_gap'
]

# Create model-ready dataset by removing rows with missing values
model_data = df[features + [target]].dropna()

# Split into X (features) and y (target)
X = model_data[features]
y = model_data[target]

# Display final count after cleaning
print(f"Final feature set contains {X.shape[0]} suburbs.")

Final feature set contains 1699 suburbs.


> **Note:** The count stays at **1,699 suburbs**. This shows our data is clean and every physical suburb has the information we need. We are not losing any more data because we already handled the non-spatial rows (like 'Offshore' and 'No Usual Address') in [01_data_cleaning.ipynb](./01_data_cleaning.ipynb).

### 1.3 Splitting the Data (Train vs. Test)
To verify that our model can predict housing stress for suburbs it has never seen before, we must split our data. We use the `train_test_split` function from the `sklearn` library to set aside 20% of our suburbs as a "Test Set." 

The model will learn only from the remaining 80% (the "Training Set"). This is a standard practice to prevent the model from simply memorising the data points.

In [14]:
from sklearn.model_selection import train_test_split

# Split data into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Display the number of suburbs in each set
print(f"Training set size: {X_train.shape[0]} suburbs")
print(f"Testing set size: {X_test.shape[0]} suburbs")

Training set size: 1359 suburbs
Testing set size: 340 suburbs


### 1.4 Feature Scaling
Machine learning models perform best when all input features are on a similar scale. Currently, variables like `avg_weekly_income` are in the thousands, while `unemployment_rate` is a small decimal. 

We use `StandardScaler` to transform these so they have a mean of 0 and a standard deviation of 1. This prevents any single variable from dominating the model due to its numerical magnitude. This step ensures that we normalise the impact of each feature.

In [17]:
from sklearn.preprocessing import StandardScaler

# Initialize the scaler
scaler = StandardScaler()

# Fit on training data and transform both sets
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Verify scaling by checking the first transformed row
print("First row of scaled training data:")
print(X_train_scaled[0])

First row of scaled training data:
[-1.42077205 -0.917771   -0.87470449 -0.48179693 -0.50768355 -2.89833658
 -1.35136137]


> **Note:** The output shows our features as a list of numbers centered around zero. For example, a value of `-1.42` means that specific suburb's feature is 1.42 standard deviations below the Western Australian average, while a positive number would be above average.

### 1.5 Section Summary
We have successfully prepared the data for modelling by:
* Selecting **7 key features** to predict housing stress.
* Confirming a clean dataset of **1,699 suburbs**.
* Splitting the data to allow for unbiased testing.
* Scaling the values to ensure the model treats all features fairly.